# GLM Model Training and Evaluation


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_validate
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("scotiabank_ml")

train = pd.read_csv("../data/splits/train.csv")

FEATURES = ["X1","X2","X3","X4","X6","X7","X8","X10","X11","X12","X3_missing","X4_missing","X9_encoded"]
TARGET = "TARGET"

X_train = train[FEATURES]
y_train = train[TARGET]

c:\Users\IRVIN\miniconda3\envs\test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ridge = Ridge(alpha=1.0)

cv_results = cross_validate(
    ridge, X_train, y_train,
    cv=5,
    scoring=["neg_root_mean_squared_error", "r2"],
    return_train_score=True
)

rmse_val   = -cv_results["test_neg_root_mean_squared_error"].mean()
rmse_std   = cv_results["test_neg_root_mean_squared_error"].std()
r2_val     = cv_results["test_r2"].mean()
r2_std     = cv_results["test_r2"].std()
rmse_train = -cv_results["train_neg_root_mean_squared_error"].mean()
r2_train   = cv_results["train_r2"].mean()

with mlflow.start_run(run_name="ridge"):
    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("val_rmse_mean", rmse_val)
    mlflow.log_metric("val_rmse_std", rmse_std)
    mlflow.log_metric("val_r2_mean", r2_val)
    mlflow.log_metric("val_r2_std", r2_std)
    mlflow.log_metric("train_rmse_mean", rmse_train)
    mlflow.log_metric("train_r2_mean", r2_train)

    # Fit on full train for artifact logging
    ridge.fit(X_train, y_train)
    mlflow.sklearn.log_model(ridge, "ridge_model")

print(f"Ridge | Val RMSE: {rmse_val:.4f} ± {rmse_std:.4f} | Val R²: {r2_val:.4f} ± {r2_std:.4f}")
print(f"Ridge | Train RMSE: {rmse_train:.4f} | Train R²: {r2_train:.4f}")

2026/06/26 01:58:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/26 01:58:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run ridge at: http://127.0.0.1:5000/#/experiments/2/runs/89878c4a0d224112a9a99fdd7c5c87d4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Ridge | Val RMSE: 0.5454 ± 0.0042 | Val R²: 0.3083 ± 0.0097
Ridge | Train RMSE: 0.5450 | Train R²: 0.3096


In [3]:
lasso = Lasso(alpha=0.001)

cv_results = cross_validate(
    lasso, X_train, y_train,
    cv=5,
    scoring=["neg_root_mean_squared_error", "r2"],
    return_train_score=True
)

rmse_val   = -cv_results["test_neg_root_mean_squared_error"].mean()
rmse_std   = cv_results["test_neg_root_mean_squared_error"].std()
r2_val     = cv_results["test_r2"].mean()
r2_std     = cv_results["test_r2"].std()
rmse_train = -cv_results["train_neg_root_mean_squared_error"].mean()
r2_train   = cv_results["train_r2"].mean()

with mlflow.start_run(run_name="lasso"):
    mlflow.log_param("model", "Lasso")
    mlflow.log_param("alpha", 0.001)
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("val_rmse_mean", rmse_val)
    mlflow.log_metric("val_rmse_std", rmse_std)
    mlflow.log_metric("val_r2_mean", r2_val)
    mlflow.log_metric("val_r2_std", r2_std)
    mlflow.log_metric("train_rmse_mean", rmse_train)
    mlflow.log_metric("train_r2_mean", r2_train)

    lasso.fit(X_train, y_train)
    mlflow.sklearn.log_model(lasso, "lasso_model")

print(f"Lasso | Val RMSE: {rmse_val:.4f} ± {rmse_std:.4f} | Val R²: {r2_val:.4f} ± {r2_std:.4f}")
print(f"Lasso | Train RMSE: {rmse_train:.4f} | Train R²: {r2_train:.4f}")

2026/06/26 01:58:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/26 01:58:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lasso at: http://127.0.0.1:5000/#/experiments/2/runs/8fdd08a7df3a473b9e633151d9569fec
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Lasso | Val RMSE: 0.5454 ± 0.0042 | Val R²: 0.3083 ± 0.0094
Lasso | Train RMSE: 0.5450 | Train R²: 0.3096
